In [ ]:
!pip install scikit-learn -q

import os
import shutil
import numpy as np
from PIL import Image, ImageOps
from torchvision import datasets
from sklearn.model_selection import StratifiedShuffleSplit
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/AML_Dataset/Warp-C"
SAVE_DIR = "/content/WaRP-C-preprocessed"

class PadToSquare:
    def __call__(self, img):
        w, h = img.size
        d = max(w, h)
        pl, pt = (d - w) // 2, (d - h) // 2
        return ImageOps.expand(img, (pl, pt, d - w - pl, d - h - pt), fill=(128, 128, 128))

In [ ]:
def list_samples(root_dir):
    #Goes through the raw WaRP-C crop folders and returns (path, class_name) pairs.
    samples, classes = [], []

    for superclass in sorted(os.listdir(root_dir)):
        superclass_path = os.path.join(root_dir, superclass)
        if not os.path.isdir(superclass_path):
            continue

        for subclass in sorted(os.listdir(superclass_path)):
            subclass_path = os.path.join(superclass_path, subclass)
            if not os.path.isdir(subclass_path):
                continue

            images_here = [f for f in os.listdir(subclass_path) if f.lower().endswith(".jpg")]

            if images_here:
                if subclass not in classes:
                    classes.append(subclass)
                for img_name in images_here:
                    samples.append((os.path.join(subclass_path, img_name), subclass))
            else:
                for subsubclass in sorted(os.listdir(subclass_path)):
                    subsubclass_path = os.path.join(subclass_path, subsubclass)
                    if not os.path.isdir(subsubclass_path):
                        continue
                    if subsubclass not in classes:
                        classes.append(subsubclass)
                    for img_name in os.listdir(subsubclass_path):
                        if img_name.lower().endswith(".jpg"):
                            samples.append((os.path.join(subsubclass_path, img_name), subsubclass))

    return samples, classes


def build_superclass_map(crops_dir):
    #Automatically builds subclass and superclass mapping from folder structure.
    mapping = {}
    for superclass in os.listdir(crops_dir):
        superclass_path = os.path.join(crops_dir, superclass)
        if not os.path.isdir(superclass_path):
            continue
        for subclass in os.listdir(superclass_path):
            subclass_path = os.path.join(superclass_path, subclass)
            if os.path.isdir(subclass_path):
                mapping[subclass] = superclass
    return mapping

In [ ]:
train_dir = f"{DRIVE_ROOT}/train_crops"
test_dir  = f"{DRIVE_ROOT}/test_crops"

train_samples, classes = list_samples(train_dir)
test_samples, _ = list_samples(test_dir)
class_to_idx = {c: i for i, c in enumerate(classes)}

print(f"Total train images : {len(train_samples)}")
print(f"Total test images  : {len(test_samples)}")
print(f"Classes ({len(classes)}): {classes}")

labels = np.array([class_to_idx[c] for _, c in train_samples])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

print(f"\ntrain: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_samples)}")

superclass_map = build_superclass_map(train_dir)

Total train images : 8823
Total test images  : 1551
Classes (28): ['bottle-blue', 'bottle-blue-full', 'bottle-blue5l', 'bottle-blue5l-full', 'bottle-dark', 'bottle-dark-full', 'bottle-green', 'bottle-green-full', 'bottle-milk', 'bottle-milk-full', 'bottle-multicolor', 'bottle-multicolorv-full', 'bottle-oil', 'bottle-oil-full', 'bottle-transp', 'bottle-transp-full', 'bottle-yogurt', 'glass-dark', 'glass-green', 'glass-transp', 'canister', 'cans', 'juice-cardboard', 'milk-cardboard', 'detergent-box', 'detergent-color', 'detergent-transparent', 'detergent-white']

train: 7058 | val: 1765 | test: 1551


In [ ]:
def save_split(split_name, samples, indices=None):
    items = samples if indices is None else [samples[i] for i in indices]
    count = 0
    for img_path, cls_name in items:
        superclass = superclass_map.get(cls_name, "other")
        out_dir = f"{SAVE_DIR}/{split_name}/{superclass}/{cls_name}"
        os.makedirs(out_dir, exist_ok=True)

        img = Image.open(img_path).convert("RGB")
        img = PadToSquare()(img)
        img = img.resize((224, 224), Image.BILINEAR)
        img.save(f"{out_dir}/{os.path.basename(img_path)}")
        count += 1
    print(f"  {split_name}: {count} images saved")
save_split("train", train_samples, train_idx)
save_split("val", train_samples, val_idx)
save_split("test", test_samples)


  train: 7058 images saved
  val: 1765 images saved
  test: 1551 images saved


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/WaRP-C-preprocessed", "zip", SAVE_DIR)
files.download("/content/WaRP-C-preprocessed.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>